# 文件操作 · 知识点

### 1、读取文件

In [ ]:
# open(path, mode, encoding=...) 三个最常用的参数：
# mode："r" 只读（文件不存在会报错 FileNotFoundError）
# encoding：用什么字符编码读写文本。显式写 "utf-8" 是好习惯——不写的话默认值跟着操作系统走
#           （Windows 上常常默认是 gbk），同一份代码换台机器跑，读中文文件就可能乱码或直接报错
with open("知识点.ipynb","r",encoding="utf-8") as f:
    # f 本身是一个"文件对象"（类型是 _io.TextIOWrapper），不是字符串，直接 print(f) 打印的是
    # 这个对象的描述，不是文件内容；要拿到内容必须通过 read()/readline() 或者下面的 for 循环迭代
    print(type(f))          # <class '_io.TextIOWrapper'>

    # for 循环会触发文件对象的迭代协议：每次循环自动取出"下一行"，转换成字符串赋给 l
    # 所以 l 已经是 str 了，能直接 print；末尾还带着原来的换行符 \n
    for l in f:
        print(type(l))      # <class 'str'>
        print(l)

In [ ]:
with open("知识点.ipynb","r",encoding="utf-8") as f:
    # read()：一次性把整个文件内容读成一个字符串，简单但文件很大时会占用大量内存
    content = f.read()
    print(content)

# 对比：
# for line in f          逐行读取，每次只在内存里放一行，适合处理大文件
# f.readlines()          一次性读成一个 list，每个元素是一行（自己再决定要不要去掉换行符）

### 2、写文件

In [ ]:
# mode="w"：文件不存在会自动创建；文件已存在会先清空再写，原内容会丢失，不是追加
# 注意：write() 不会像 print() 一样自动加换行符，多行内容要自己在字符串里加 "\n"
with open("write.txt","w",encoding="utf-8") as w :
    w.write("Hello Python")

### 3、文件追加内容

In [ ]:
# mode="a"：在文件末尾追加内容，不会清空原内容；文件不存在则自动创建
with open("write.txt","a",encoding="utf-8") as ap:
    ap.write("追加的内容")

### 4、文件操作

In [ ]:
from pathlib import Path

p = Path("docs/test.txt")
print(p.exists())       # 路径是否存在（文件或目录都行）
print(p.parent)         # 父目录路径，docs
print(p.name)           # 文件名（带后缀），test.txt
print(p.stem)           # 文件名（不带后缀），test
print(p.suffix)         # 后缀名，.txt

# 下面两个方法要求路径真实存在，否则会报错：
# p.read_text()         直接读取文本内容并返回字符串，等价于 open(p).read()，不用自己管 with/close
# p.unlink()             删除这个文件；文件不存在会报 FileNotFoundError
#                        如果不确定文件是否存在又不想报错，用 p.unlink(missing_ok=True)

### 5、目录操作

In [ ]:
from pathlib import Path

# mkdir() 两个关键参数：
# parents=True   父目录不存在时一并自动创建（类似 shell 的 mkdir -p）；默认 False，父目录不存在会报错 FileNotFoundError
# exist_ok=True  目录已存在时不报错，静默跳过；默认 False，已存在会报错 FileExistsError
# Path("new_file").mkdir()
# Path("a/b/c").mkdir(parents=True, exist_ok=True)
path = Path("new_file")
# 删除目录：rmdir() 只能删空目录，目录里还有内容会报错
path.rmdir()

p = Path("folder")
for f in p.iterdir():       # 遍历目录下所有文件和子目录，只有一层，不会进子目录
    print(f.name)

# glob() 的匹配模式（通配符）：
# *   匹配任意数量的任意字符（不跨目录层级）  例如 *.txt 匹配当前层所有 txt 文件
# ?   匹配单个任意字符
# **  和 rglob()（见第 12 节）配合才会递归匹配多层目录
for f in p.glob("*.txt"):   # 只遍历当前层的 txt 文件
    print(f.name)

### 6、JSON文件

In [ ]:
# 读取json
import json

with open("config.json","r",encoding="utf-8") as f:
    data = json.load(f)
    print(data)

# 直接解析json格式的字符串（不经过文件）
data2 = '{"name": "Alice", "age": 30}'
result = json.loads(data2)
print(result)

# 如果文件内容不是合法的 JSON（比如手写错了逗号、引号），json.load/loads 会报 json.JSONDecodeError
# 这是 ValueError 的子类，处理异常时记得对应上（第 7 节会用到）

In [ ]:
# 写入JSON
import json

data = {"name": "Alice", "age": 30}

with open("config.json","w",encoding="utf-8") as f:
    # json.dump(obj, fp, **kwargs) 把 obj 序列化成 JSON 文本写进文件，常用两个参数：
    # ensure_ascii：默认 True，会把非 ASCII 字符（比如中文）转义成 \uXXXX 这种形式存进文件；
    #               设成 False 才会保留中文原样，文件里直接显示「张三」而不是「\u5f20\u4e09」
    # indent：缩进的空格数，用来格式化输出方便人眼阅读；不写的话默认压缩成一行，没有换行和缩进
    # （对应的反序列化函数 json.load 没有这两个参数，因为读取不需要关心排版）
    json.dump(data,f,ensure_ascii=False,indent=2)

### 7、异常处理

In [ ]:
# try:
    # 可能出错的代码
# except Exception as e:
    # 处理异常
# finally:
    # 一定会执行

# except 要按从具体到笼统的顺序写：Python 按顺序匹配，第一个匹配上的 except 生效，后面的不会再看
# FileNotFoundError、PermissionError 都是 OSError 的子类，OSError 又是 Exception 的子类
# 如果把 except Exception 写在前面，下面更具体的 except 永远不会被触发（顺序写反不会报错，但分支永远死代码）
try:
    with open("test.txt", "r") as f:
        data = f.read()
except FileNotFoundError as e:
    print(f"文件不存在：{e}")
except PermissionError as e:
    print(f"没有权限：{e}")
except Exception as e:
    print(f"其他错误：{e}")

### 8、JSONL 文件

In [ ]:
# JSONL（JSON Lines）：每行一个独立的 JSON 对象，用换行分隔，不是一个大数组
# 适合逐行处理大文件 / 流式日志，不需要一次性把整个文件读进内存
import json

records = [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]
with open("data.jsonl", "w", encoding="utf-8") as f:
    for r in records:
        # json.dumps（带 s）：把对象转成 JSON 字符串，不写文件，区别于 json.dump（直接写文件）
        # ensure_ascii=False 含义同上面 json.dump：保留中文原样，不转义成 \uXXXX
        # JSONL 没有内置的批量写入函数，要自己逐行 write，并手动在每行末尾加换行符 "\n"
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [ ]:
import json

# 逐行读取 JSONL，每行单独 json.loads，内存友好
with open("data.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        print(record)

### 9、CSV 文件

In [ ]:
import csv

# writer/reader：按行处理，行是 list，没有表头概念
rows = [["name", "age"], ["Alice", 30], ["Bob", 25]]
with open("users.csv", "w", encoding="utf-8", newline="") as f:
    # newline="" 控制的是文本模式下换行符的自动翻译：
    # 默认 newline=None 时，Python 会把你写入的 \n 自动翻译成系统换行符（Windows 上是 \r\n）
    # 但 csv 模块内部已经自己按标准写了 \r\n 作为行结尾，如果再叠加一次系统翻译，
    # Windows 上就会变成 \r\r\n，每行之间多出一个空行——newline="" 关掉这层翻译，避免这个问题
    # delimiter 默认是逗号 ","，改成 "\t" 就能写出 TSV（Tab 分隔）文件
    writer = csv.writer(f)
    writer.writerows(rows)

# 读 csv 同样建议加 newline=""：不加的话，如果某个字段内容本身包含换行符，解析可能会出错
with open("users.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

In [3]:
import csv

# DictWriter/DictReader：按字段名读写，更贴近实际数据结构（有表头）
with open("users2.csv", "w", encoding="utf-8", newline="") as f:
    # fieldnames 是必填参数：决定表头的列名和写入顺序
    # writerow() 传入的字典，key 必须能在 fieldnames 里找到——多出的 key 会报错，缺的 key 会写成空值
    writer = csv.DictWriter(f, fieldnames=["name", "age"])
    writer.writeheader()        # 按 fieldnames 写入第一行表头
    writer.writerow({"name": "Alice", "age": 30})
    writer.writerow({"name": "Bob", "age": 25})

with open("users2.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)  # 自动把第一行当表头，后面每行转成 {表头: 值} 的字典
    for row in reader:
        print(row)   # {'name': 'Alice', 'age': '30'}  注意：所有值都是字符串

{'name': 'Alice', 'age': '30'}
{'name': 'Bob', 'age': '25'}


### 10、编码处理

In [ ]:
# 编码不匹配是文件操作中最常见的坑：utf-8 是默认推荐编码，gbk/gb2312 是 Windows 中文环境常见编码
# 常见编码：utf-8（推荐，国际通用，中英文都用 1~3 字节）/ gbk（windows 简体中文老系统）/ ascii（仅英文数字符号）
# 用错误的编码打开文件会触发 UnicodeDecodeError
try:
    with open("gbk_file.txt", "r", encoding="utf-8") as f:
        f.read()
except UnicodeDecodeError as e:
    print(f"编码错误：{e}")

# 真正的解法是换成正确编码，不是靠 errors 参数硬读
with open("gbk_file.txt", "r", encoding="gbk") as f:
    content = f.read()

In [ ]:
# errors 参数：遇到无法解码的字节时如何处理
# 'strict'（默认，报错）/ 'ignore'（直接丢弃非法字节）/ 'replace'（替换成 �）
# 注意：errors='ignore'/'replace' 只是让程序不崩，并不会让内容变正确——文字该乱码还是乱码、该丢的字节还是丢了
# 真正解决编码问题永远是换成文件本身的编码（上面的 encoding="gbk"），errors 只在编码未知、能接受丢字符时兜底用
with open("gbk_file.txt", "r", encoding="utf-8", errors="ignore") as f:
    content = f.read()

### 11、io.StringIO

In [ ]:
import io

# 在内存里模拟一个"文件对象"，不需要真正落盘
buf = io.StringIO()
buf.write("Hello ")
buf.write("World")
# 写完之后游标停在末尾，直接 buf.read() 会读到空字符串——这是新手常踩的坑
# 要读取已写入的内容，要么用 getvalue()（不影响游标），要么 buf.seek(0) 把游标移回开头再 read()
print(buf.getvalue())   # Hello World

buf2 = io.StringIO("line1\nline2\nline3")
for line in buf2:       # StringIO 也能像文件一样逐行迭代
    print(line.strip())

# StringIO 处理文本（str），如果是二进制数据（bytes）要用 io.BytesIO，用法类似

In [ ]:
import io, csv

# 典型用途：把字符串伪装成文件，喂给只接受文件对象的 API（如 csv.reader）
csv_text = "name,age\nAlice,30\nBob,25"
reader = csv.DictReader(io.StringIO(csv_text))
for row in reader:
    print(row)

### 12、递归遍历目录

In [ ]:
from pathlib import Path

# iterdir()/glob() 只遍历一层，rglob() 递归遍历所有子目录
p = Path(".")
for f in p.rglob("*.py"):
    print(f)

In [ ]:
import os

# os.walk：经典写法，逐层返回 (当前目录, 子目录列表, 文件列表)
for dirpath, dirnames, filenames in os.walk("."):
    for name in filenames:
        print(os.path.join(dirpath, name))

### 13、.env 环境变量

In [ ]:
import os

# 读取系统环境变量，不需要额外库
db_host = os.environ.get("DB_HOST", "localhost")   # 不存在时返回默认值
db_host2 = os.getenv("DB_HOST", "localhost")        # 等价写法

In [ ]:
# .env 文件把配置（数据库密码、API Key 等敏感信息）放到代码外部，不提交到 git
# 运行时加载进环境变量，对比 Java：类似 application.properties + System.getenv() 的组合
# .env 本身没有内建支持，约定俗成需要 python-dotenv 库（pip install python-dotenv）

# .env 文件内容示例（放在项目根目录）：
# DB_HOST=127.0.0.1
# DB_PASSWORD=secret123

from dotenv import load_dotenv
import os

# load_dotenv(dotenv_path=None, override=False) 两个常用参数：
# dotenv_path：指定 .env 文件路径，不写默认从当前工作目录开始往上找 .env
# override：默认 False —— 如果系统里已经存在同名环境变量，.env 里的值不会覆盖它
#           （常见坑：改了 .env 但读出来的值没变，往往是外部 shell 已经 export 过同名变量）
load_dotenv()
print(os.getenv("DB_HOST"))